In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
C:\Users\User\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class JokeState(TypedDict):
    
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState) -> dict:
    
    prompt = f"Generate a joke on the topic {state["topic"]}"
    response = llm.invoke(prompt).content
    
    return {
        'joke': response
    }

In [6]:
def generate_explanation(state: JokeState) -> dict:
    
    prompt = f"Write an explanation for the joke - {state['joke']}"
    
    response = llm.invoke(prompt).content
    
    return {
        'explanation': response
    }

In [8]:

graph = StateGraph(JokeState)


# Nodes
graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)


# Edges
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

# Checkpointer
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [23]:
# Thread-1
config1 = {
    "configurable": {
        "thread_id": "1"
    }
}

# Thread-2
config2 = {
    "configurable": {
        "thread_id": "2"
    }
}

response1 = workflow.invoke({
    'topic': 'pizza'
}, config=config1)

response2 = workflow.invoke({
    'topic': 'pasta'
}, config=config2)


In [24]:
response1

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crust-y!',
 'explanation': 'This joke plays with the double meaning of the word "crust." In one sense, "crust" refers to the outer layer of a pizza. However, it is also used informally to describe someone who is grumpy or irritable. So, in this joke, the pizza goes to the doctor because it is feeling "crust-y" or irritable, rather than because it has a physical ailment.'}

In [25]:
response2

{'topic': 'pasta',
 'joke': 'Why did the spaghetti go to the party? \n\nBecause it heard it was going to be a pasta-tively fun time!',
 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the word "positively" but with "pasta" in it, referencing the fact that spaghetti is a type of pasta. The joke suggests that the spaghetti went to the party because it heard it was going to be a fun time, using a pun to make the punchline humorous. Overall, it\'s a light-hearted and punny joke about a food item attending a party.'}

In [26]:
# To get the state of config1
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crust-y!', 'explanation': 'This joke plays with the double meaning of the word "crust." In one sense, "crust" refers to the outer layer of a pizza. However, it is also used informally to describe someone who is grumpy or irritable. So, in this joke, the pizza goes to the doctor because it is feeling "crust-y" or irritable, rather than because it has a physical ailment.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-0c31-65e9-801a-5fae3193294a'}}, metadata={'source': 'loop', 'step': 26, 'parents': {}}, created_at='2026-07-05T07:27:12.768718+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-01ce-6c94-8019-64e43872129a'}}, tasks=(), interrupts=())

In [27]:
# To get the state of config2
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to the party? \n\nBecause it heard it was going to be a pasta-tively fun time!', 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the word "positively" but with "pasta" in it, referencing the fact that spaghetti is a type of pasta. The joke suggests that the spaghetti went to the party because it heard it was going to be a fun time, using a pun to make the punchline humorous. Overall, it\'s a light-hearted and punny joke about a food item attending a party.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-2be4-67a7-8006-990be9d320e3'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-05T07:27:16.092693+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-1e19-65da-8005-1bf91f12ba2e'}}, tasks=(), interrupts=())

In [28]:
# Intermediate State History of config1
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a little crust-y!', 'explanation': 'This joke plays with the double meaning of the word "crust." In one sense, "crust" refers to the outer layer of a pizza. However, it is also used informally to describe someone who is grumpy or irritable. So, in this joke, the pizza goes to the doctor because it is feeling "crust-y" or irritable, rather than because it has a physical ailment.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-0c31-65e9-801a-5fae3193294a'}}, metadata={'source': 'loop', 'step': 26, 'parents': {}}, created_at='2026-07-05T07:27:12.768718+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-01ce-6c94-8019-64e43872129a'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? Because it was feeling a littl

In [29]:
# Intermediate State History of config2
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti go to the party? \n\nBecause it heard it was going to be a pasta-tively fun time!', 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the word "positively" but with "pasta" in it, referencing the fact that spaghetti is a type of pasta. The joke suggests that the spaghetti went to the party because it heard it was going to be a fun time, using a pun to make the punchline humorous. Overall, it\'s a light-hearted and punny joke about a food item attending a party.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-2be4-67a7-8006-990be9d320e3'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-05T07:27:16.092693+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17842f-1e19-65da-8005-1bf91f12ba2e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 

FOR COMPLEX WORKFLOW (EVERYTHING BELOW THIS POINT)

TIME TRAVEL

In [30]:
workflow.get_state({'configurable': {'thread_id': '2', "checkpoint_id": "1f178426-d216-6cc8-8000-a117524f938d"}})

StateSnapshot(values={'topic': 'pasta'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f178426-d216-6cc8-8000-a117524f938d'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-05T07:23:31.927757+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f178426-d210-6f35-bfff-7f327d4256bf'}}, tasks=(PregelTask(id='ee860b4f-912c-1686-eb80-5e1fcc3ba565', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pasta chef break up with his girlfriend?\n\nBecause she was too saucy for him!'}),), interrupts=())

Now start the execution again from this checkpoint_id

In [31]:
workflow.invoke(None, {'configurable': {'thread_id': '2', "checkpoint_id": "1f178426-d216-6cc8-8000-a117524f938d"}})

{'topic': 'pasta',
 'joke': 'Why do cannibals never eat pasta? \n\nBecause they prefer finger food!',
 'explanation': 'This joke plays on the fact that cannibals are known for eating human flesh. The punchline "because they prefer finger food" is a play on words, as the term "finger food" usually refers to small, easily eaten foods that can be consumed with the fingers. In this case, it humorously suggests that cannibals prefer to eat fingers, a common body part they might consume if they were to eat human flesh. The joke is meant to be dark humor, playing on the macabre nature of cannibalism in a lighthearted and playful way.'}

In [32]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why do cannibals never eat pasta? \n\nBecause they prefer finger food!', 'explanation': 'This joke plays on the fact that cannibals are known for eating human flesh. The punchline "because they prefer finger food" is a play on words, as the term "finger food" usually refers to small, easily eaten foods that can be consumed with the fingers. In this case, it humorously suggests that cannibals prefer to eat fingers, a common body part they might consume if they were to eat human flesh. The joke is meant to be dark humor, playing on the macabre nature of cannibalism in a lighthearted and playful way.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17859e-2133-69d3-8003-b4edd0ac35bd'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-05T10:11:26.552914+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17859e-0b7a-6c0d-

Update State (change topic -> samosa)

In [33]:
workflow.update_state({'configurable': {'thread_id': '2', "checkpoint_id": "1f178426-d216-6cc8-8000-a117524f938d", "checkpoint_ns": ""}}, {'topic': 'samosa'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1785a7-07a1-6009-8001-e0409f76f272'}}

In [34]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1785a7-07a1-6009-8001-e0409f76f272'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-05T10:15:25.463320+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f178426-d216-6cc8-8000-a117524f938d'}}, tasks=(PregelTask(id='97cf5d3f-1ff0-a4de-9db3-71b964436159', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why do cannibals never eat pasta? \n\nBecause they prefer finger food!', 'explanation': 'This joke plays on the fact that cannibals are known for eating human flesh. The punchline "because they prefer finger food" is a play on words, as the term "finger food" usually refers to small, easily eaten foods that can be consumed with 

Executing the workflow again with that updated state (changed topic -> samosa) by its respective checkpoint_id

In [35]:
workflow.invoke(None, {'configurable': {'thread_id': '2', "checkpoint_id": "1f1785a7-07a1-6009-8001-e0409f76f272"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school?\nBecause it wanted to be a little more well-rounded!',
 'explanation': 'This joke plays on the word "well-rounded," which can refer to someone who is knowledgeable in many different areas or well-proportioned physically. In this case, the joke is making a pun by implying that the samosa, a triangular fried pastry popular in Indian cuisine, wanted to become more well-rounded in shape like a circle by attending school. It\'s a light-hearted play on words that adds humor to the idea of a samosa attending school for self-improvement.'}

In [36]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to school?\nBecause it wanted to be a little more well-rounded!', 'explanation': 'This joke plays on the word "well-rounded," which can refer to someone who is knowledgeable in many different areas or well-proportioned physically. In this case, the joke is making a pun by implying that the samosa, a triangular fried pastry popular in Indian cuisine, wanted to become more well-rounded in shape like a circle by attending school. It\'s a light-hearted play on words that adds humor to the idea of a samosa attending school for self-improvement.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1785ad-5d1a-62ed-8003-58fc8c077f2d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-05T10:18:15.487133+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1785ad-46f1-6ea0-8002-8f3ece3b87bb'}}, tasks=(), inte